In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Transform: converting images to tensors and normalize
transform = transforms.Compose([
    transforms.ToTensor(),                               # Converts PIL image to tensor
    transforms.Normalize((0.1307,),(0.3081,))            # Normalize with MNIST mean and std where the values are previously are calculated.
])

# MNIST dataset 
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

In [14]:
(X_train, y_train) = train_dataset.data, train_dataset.targets
(X_test, y_test) = test_dataset.data, test_dataset.targets

X_train = X_train / 255.0
X_test = X_test / 255.0

encoder = OneHotEncoder(sparse_output=False)
y_train_oh = encoder.fit_transform(y_train.reshape(-1, 1))
y_test_oh = encoder.transform(y_test.reshape(-1, 1))

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)
print("Y Training data shape:", y_train.shape)
print("Y Testing data shape:", y_test.shape)

Training data shape: torch.Size([60000, 28, 28])
Testing data shape: torch.Size([10000, 28, 28])
Y Training data shape: torch.Size([60000])
Y Testing data shape: torch.Size([10000])


In [23]:
input_layer, hidden_layer, output_layer, lr = 784, 128, 10, 0.7
np.random.seed(42)

# 1. Force everything to pure NumPy immediately
X_train_flat = np.array(X_train).reshape(-1, 784).astype(np.float32)
y_train_oh = np.array(y_train_oh).astype(np.float32)

# 2. Re-initialize weights to ensure they are float32 NumPy arrays
W1 = (np.random.randn(input_layer, hidden_layer) * np.sqrt(2. / input_layer)).astype(np.float32)
b1 = np.zeros((1, hidden_layer), dtype=np.float32)
W2 = (np.random.randn(hidden_layer, output_layer) * np.sqrt(2. / hidden_layer)).astype(np.float32)
b2 = np.zeros((1, output_layer), dtype=np.float32)

# Ensure weights are numpy (they should be if you used np.random)
W1 = np.array(W1)
b1 = np.array(b1)
W2 = np.array(W2)
b2 = np.array(b2)

for epoch in range(500):
    # --- Forward Pass ---
    # 1. Linear 1
    z1 = np.dot(X_train_flat, W1) + b1  # Use np.dot for clarity or @

    # 2. ReLU
    a1 = np.maximum(0, z1)   

    # 3. Linear 2
    z2 = np.dot(a1, W2) + b2

    # 4. Stable Softmax
    # Subtracting the max per row for numerical stability
    z2_max = np.max(z2, axis=1, keepdims=True)
    exp_scores = np.exp(z2 - z2_max)
    a2 = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

    # --- Backward Pass ---
    m = y_train_oh.shape[0]

    dz2 = a2 - y_train_oh # Derivative of Cross-Entropy + Softmax is surprisingly simple!
    dW2 = (a1.T @ dz2) / m
    db2 = np.sum(dz2, axis=0, keepdims=True) / m

    da1 = dz2 @ W2.T
    dz1 = da1 * (z1 > 0) # Derivative of ReLU
    dW1 = (X_train_flat.T @ dz1) / m
    db1 = np.sum(dz1, axis=0, keepdims=True) / m

    # --- Update ---
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

    if epoch % 10 == 0:
        loss = -np.sum(y_train_oh * np.log(a2 + 1e-8)) / m
        # Calculate accuracy
        predictions = np.argmax(a2, axis=1)
        labels = np.argmax(y_train_oh, axis=1)
        accuracy = np.mean(predictions == labels)
        print(f'Epoch {epoch}, Loss: {loss:.4f}, Accuracy: {accuracy:.2%}')

C:\Users\amogh\AppData\Local\Temp\ipykernel_47232\1849497515.py:5: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  X_train_flat = np.array(X_train).reshape(-1, 784).astype(np.float32)


Epoch 0, Loss: 2.4117, Accuracy: 12.48%
Epoch 10, Loss: 0.7759, Accuracy: 77.51%
Epoch 20, Loss: 0.7987, Accuracy: 72.29%
Epoch 30, Loss: 0.4239, Accuracy: 87.10%
Epoch 40, Loss: 0.3559, Accuracy: 89.84%
Epoch 50, Loss: 0.3606, Accuracy: 89.24%
Epoch 60, Loss: 0.2954, Accuracy: 91.52%
Epoch 70, Loss: 0.2777, Accuracy: 92.01%
Epoch 80, Loss: 0.2633, Accuracy: 92.41%
Epoch 90, Loss: 0.2509, Accuracy: 92.84%
Epoch 100, Loss: 0.2400, Accuracy: 93.15%
Epoch 110, Loss: 0.2302, Accuracy: 93.45%
Epoch 120, Loss: 0.2214, Accuracy: 93.69%
Epoch 130, Loss: 0.2134, Accuracy: 93.94%
Epoch 140, Loss: 0.2061, Accuracy: 94.13%
Epoch 150, Loss: 0.1994, Accuracy: 94.34%
Epoch 160, Loss: 0.1932, Accuracy: 94.53%
Epoch 170, Loss: 0.1874, Accuracy: 94.70%
Epoch 180, Loss: 0.1821, Accuracy: 94.85%
Epoch 190, Loss: 0.1770, Accuracy: 94.97%
Epoch 200, Loss: 0.1723, Accuracy: 95.09%
Epoch 210, Loss: 0.1679, Accuracy: 95.23%
Epoch 220, Loss: 0.1637, Accuracy: 95.33%
Epoch 230, Loss: 0.1598, Accuracy: 95.45%
Epo

##### USING PYTORCH

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Transform: converting images to tensors and normalize
transform = transforms.Compose([
    transforms.ToTensor(),                               # Converts PIL image to tensor
    transforms.Normalize((0.1307,),(0.3081,))            # Normalize with MNIST mean and std where the values are previously are calculated.
])

# MNIST dataset 
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

(X_train, y_train) = train_dataset.data, train_dataset.targets
(X_test, y_test) = test_dataset.data, test_dataset.targets

X_train = X_train / 255.0
X_test = X_test / 255.0

encoder = OneHotEncoder(sparse_output=False)
y_train_oh = encoder.fit_transform(y_train.reshape(-1, 1))
y_test_oh = encoder.transform(y_test.reshape(-1, 1))

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)
print("Y Training data shape:", y_train.shape)
print("Y Testing data shape:", y_test.shape)


Training data shape: torch.Size([60000, 28, 28])
Testing data shape: torch.Size([10000, 28, 28])
Y Training data shape: torch.Size([60000])
Y Testing data shape: torch.Size([10000])


In [8]:
input_layer, hidden_layer, output_layer, lr = 784, 128, 10, 0.01
np.random.seed(42)

def mlp(input_dim=784, hidden_dims=[128], output_dim=10):
    layers = []
    prev_dim = input_dim
    for h in hidden_dims:
        layers.append(nn.Linear(prev_dim, h))
        layers.append(nn.ReLU())
        prev_dim = h
    layers.append(nn.Linear(prev_dim, output_dim))
    return nn.Sequential(*layers)

model = mlp(input_layer, [hidden_layer], output_layer)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

# Convert to torch tensors
X_train = X_train.float()
y_train = y_train.long()
X_test = X_test.float()
y_test = y_test.long()

for epoch in range(100):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train.view(X_train.size(0), -1))
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0:
        _, predicted = torch.max(outputs.data, 1)
        accuracy = (predicted == y_train).float().mean()
        print(f'Epoch {epoch+1}, Loss: {loss.item():.4f}, Accuracy: {accuracy:.2%}')

Epoch 10, Loss: 0.4147, Accuracy: 88.06%
Epoch 20, Loss: 0.3223, Accuracy: 91.35%
Epoch 30, Loss: 0.2555, Accuracy: 92.86%
Epoch 40, Loss: 0.2044, Accuracy: 94.14%
Epoch 50, Loss: 0.1694, Accuracy: 95.18%
Epoch 60, Loss: 0.1435, Accuracy: 95.90%
Epoch 70, Loss: 0.1230, Accuracy: 96.53%
Epoch 80, Loss: 0.1062, Accuracy: 97.03%
Epoch 90, Loss: 0.0921, Accuracy: 97.42%
Epoch 100, Loss: 0.0804, Accuracy: 97.71%


In [7]:
from PIL import Image
import numpy as np
import torch
from torchvision import transforms

# 1) Open image and convert explicitly to 8‑bit grayscale ("L")
def load_mnist_like(path, invert=True):
    img = Image.open(path).convert("L")   # force grayscale [web:29][web:33]

    # 2) Resize to 28x28
    img = img.resize((28, 28))

    # 3) Convert to numpy, scale to [0,1]
    img_arr = np.array(img).astype("float32") / 255.0

    # 4) If background is white and digit is dark, invert to match MNIST
    #    MNIST: background ~0 (black), digit ~1 (white) [web:38]
    if invert:
        img_arr = 1.0 - img_arr

    # 5) To tensor: shape (1, 28, 28)
    img_tensor = torch.from_numpy(img_arr).unsqueeze(0)  # add channel
    return img_tensor

# Your trained model:
# model = mlp(input_layer, [hidden_layer], output_layer)
# (already trained)

model.eval()

def predict_custom_image(path, model, invert=True):
    img_tensor = load_mnist_like(path, invert=invert)  # (1, 28, 28)
    img_tensor = img_tensor.view(1, -1)                # (1, 784)

    with torch.no_grad():
        outputs = model(img_tensor)
        _, predicted = torch.max(outputs, 1)
    return predicted.item()

# Example usage:
img_path = "1.png"
pred = predict_custom_image(img_path, model, invert=False)
print("Predicted digit:", pred)


Predicted digit: 1
